# FarmGemma: Fine-tuning Gemma 3 for Indian Agriculture

This notebook demonstrates how to fine-tune Gemma 3 for agricultural advisory tasks.

## Contents

1. [Setup & Installation](#1-setup--installation)
2. [Data Preparation](#2-data-preparation)
3. [Continuous Pre-training (CPT)](#3-continuous-pre-training-cpt)
4. [Vision Encoder Fine-tuning](#4-vision-encoder-fine-tuning)
5. [Multimodal Alignment](#5-multimodal-alignment)
6. [Instruction Fine-tuning (SFT)](#6-instruction-fine-tuning-sft)
7. [Model Export](#7-model-export)

## 1. Setup & Installation

```python
!pip install -q transformers datasets peft accelerate bitsandbytes
!pip install -q sentencepiece protobuf google-generativeai
!pip install -q torch torchvision torchaudio
```

In [ ]:
import os
import json
import torch
from transformers import (
    GemmaForCausalLM,
    GemmaTokenizer,
    AutoProcessor,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, TaskType

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Data Preparation

### 2.1 Load Agricultural Knowledge Base

```python
# Load ICAR research papers and agricultural corpus
def load_agriculture_corpus(corpus_path):
    """Load text corpus for continuous pre-training."""
    documents = []
    for root, dirs, files in os.walk(corpus_path):
        for file in files:
            if file.endswith(('.txt', '.pdf', '.json')):
                filepath = os.path.join(root, file)
                with open(filepath, 'r', encoding='utf-8') as f:
                    documents.append(f.read())
    return documents

# Load multilingual Q&A pairs
def load_qa_pairs(qa_dir, language='hi'):
    """Load Q&A pairs for instruction fine-tuning."""
    qa_file = os.path.join(qa_dir, f'{language}_qa.jsonl')
    pairs = []
    with open(qa_file, 'r', encoding='utf-8') as f:
        for line in f:
            pairs.append(json.loads(line))
    return pairs
```

In [ ]:
# Example: Load crop disease dataset
from datasets import load_dataset, ImageDataset

# For demonstration - replace with actual data paths
crop_disease_structure = {
    'image_path': 'data/crop_diseases/rice_blast.jpg',
    'disease': 'Rice Blast',
    'scientific_name': 'Magnaporthe oryzae',
    'symptoms': 'Diamond-shaped lesions on leaves',
    'treatment': 'Apply Tricyclazole 75% WP @ 0.6 g/L',
    'prevention': 'Use resistant varieties, balance nutrition'
}

print("Crop Disease Example:")
print(json.dumps(crop_disease_structure, indent=2))

## 3. Continuous Pre-training (CPT)

Pre-train Gemma on agricultural corpus to inject domain knowledge.

In [ ]:
def setup_cpt_model(model_name="google/gemma-3-4b"):
    """Initialize model for continuous pre-training."""
    model = GemmaForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    
    # Configure LoRA for efficient fine-tuning
    lora_config = LoraConfig(
        r=64,
        lora_alpha=128,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )
    
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    
    return model

# Initialize CPT model
cpt_model = setup_cpt_model()
print("\nCPT model ready for agricultural domain pre-training")

## 4. Vision Encoder Fine-tuning

Fine-tune SigLIP on crop disease and pest images.

In [ ]:
def setup_vision_model(vision_model_name="google/siglip-so400m"):
    """Initialize vision encoder for fine-tuning."""
    from transformers import AutoModel
    
    vision_encoder = AutoModel.from_pretrained(
        vision_model_name,
        torch_dtype=torch.bfloat16,
    )
    
    # Configure LoRA for vision model
    vision_lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.FEATURE_EXTRACTION
    )
    
    vision_encoder = get_peft_model(vision_encoder, vision_lora_config)
    vision_encoder.print_trainable_parameters()
    
    return vision_encoder

# Initialize vision encoder
vision_encoder = setup_vision_model()
print("\nVision encoder ready for FarmSigLIP fine-tuning")

## 5. Multimodal Alignment

Align vision features with language representations.

In [ ]:
class FarmGemmaMultimodal(torch.nn.Module):
    """Multimodal FarmGemma model combining vision and language."""
    
    def __init__(self, vision_encoder, language_model):
        super().__init__()
        self.vision_encoder = vision_encoder
        self.language_model = language_model
        self.vision_projection = torch.nn.Linear(
            vision_encoder.config.hidden_size,
            language_model.config.hidden_size
        )
    
    def forward(self, images, input_ids, attention_mask=None):
        # Get vision features
        vision_outputs = self.vision_encoder(images)
        vision_features = vision_outputs.last_hidden_state
        vision_features = self.vision_projection(vision_features)
        
        # Get language features
        language_outputs = self.language_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        return language_outputs

# Example multimodal model
farmgemma_multimodal = FarmGemmaMultimodal(vision_encoder, cpt_model)
print("\nMultimodal FarmGemma model created")

## 6. Instruction Fine-tuning (SFT)

Fine-tune on multilingual agricultural Q&A pairs.

In [ ]:
def format_instruction_sample(sample, language='en'):
    """Format Q&A sample for instruction tuning."""
    
    prompt_templates = {
        'en': """You are FarmGemma, an AI assistant for Indian farmers.

Question: {question}

Answer: {answer}""",
        'hi': """आप फार्मजेम्मा हैं, भारतीय किसानों के लिए AI सहायक।

प्रश्न: {question}

उत्तर: {answer}""",
        'ta': """நீங்கள் ஃபார்ம்ஜெம்மா, இந்திய விவசாயிகளுக்கான AI உதவியாளர்.

கேள்வி: {question}

பதில்: {answer}"""
    }
    
    template = prompt_templates.get(language, prompt_templates['en'])
    return template.format(
        question=sample['question'],
        answer=sample['answer']
    )

# Example instruction sample
sample_qa = {
    'question': 'My rice crop has brown spots on leaves. What disease is this?',
    'answer': 'Based on your description, this appears to be Rice Blast (Magnaporthe oryzae). Apply Tricyclazole 75% WP @ 0.6 g/L of water. For prevention, use resistant varieties like IR-64, Pusa-44.'
}

formatted = format_instruction_sample(sample_qa, 'en')
print("Formatted Instruction Sample:")
print(formatted)

In [ ]:
def setup_sft_training(model, output_dir="./farmgemma-sft"):
    """Configure SFT training arguments."""
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        learning_rate=2e-5,
        warmup_steps=100,
        logging_steps=50,
        save_steps=500,
        bf16=True,
        dataloader_num_workers=4,
        remove_unused_columns=False,
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,  # Load your dataset here
        data_collator=data_collator,
    )
    
    return trainer

print("\nSFT training configuration ready")

## 7. Model Export

Export for deployment on various platforms.

In [ ]:
def export_for_deployment(model, output_dir="./farmgemma-export"):
    """Export model for different deployment targets."""
    
    # 1. PyTorch checkpoint
    model.save_pretrained(f"{output_dir}/pytorch")
    
    # 2. ONNX for cross-platform deployment
    # from optimum.onnxruntime import ORTModelForCausalLM
    # ort_model = ORTModelForCausalLM.from_pretrained(model, export=True)
    # ort_model.save_pretrained(f"{output_dir}/onnx")
    
    # 3. TFLite for Android
    # from transformers import TFLiteModel
    # tflite_model = TFLiteModel.from_pretrained(model)
    # tflite_model.save_pretrained(f"{output_dir}/tflite")
    
    # 4. Quantized for edge devices
    # from transformers import BitsAndBytesConfig
    # quantization_config = BitsAndBytesConfig(load_in_4bit=True)
    
    print(f"Model exported to {output_dir}")
    print(f"  - PyTorch: {output_dir}/pytorch")
    print(f"  - ONNX: {output_dir}/onnx")
    print(f"  - TFLite: {output_dir}/tflite")
    print(f"  - Quantized: {output_dir}/quantized")

export_for_deployment(cpt_model)
print("\nExport configuration ready")

## Next Steps

1. **Data Collection**: Partner with ICAR and collect agricultural datasets
2. **Training**: Run the full training pipeline on GPU/TPU resources
3. **Evaluation**: Test with agricultural extension officers and farmers
4. **Deployment**: Build Android app and WhatsApp bot integration

## References

- [Gemma Model Card](https://ai.google.dev/gemma)
- [SigLIP Paper](https://arxiv.org/abs/2311.16982)
- [PEFT Library](https://github.com/huggingface/peft)
- [ICAR Research Portal](https://icar.gov.in/)